# Lecture 4 — Matrix games by linear programming

Companion to [Lecture 4](https://tomaskroupa.github.io/quarto-game-theory-course/lectures/04-two-player-zero-sum-games.html#sec-zero-sum-lp), using [CVXPY](https://www.cvxpy.org/).
In Colab, select **Python 3 / CPU** and **Run all**.

[Morra](https://tomaskroupa.github.io/quarto-game-theory-course/lectures/04-two-player-zero-sum-games.html#sec-zero-sum-morra): nine strategies per player. Rows = Alice, columns = Bob; both use the order `[1-1, 1-2, 1-3, 2-1, 2-2, 2-3, 3-1, 3-2, 3-3]` from the lecture.


In [ ]:
%pip -q install cvxpy
import cvxpy as cp
import numpy as np

A = np.array([[ 0,  2,  2, -3,  0,  0, -4,  0,  0],
              [-2,  0,  0,  0,  3,  3, -4,  0,  0],
              [-2,  0,  0, -3,  0,  0,  0,  4,  4],
              [ 3,  0,  3,  0, -4,  0,  0, -5,  0],
              [ 0, -3,  0,  4,  0,  4,  0, -5,  0],
              [ 0, -3,  0,  0, -4,  0,  5,  0,  5],
              [ 4,  4,  0,  0,  0, -5,  0,  0, -6],
              [ 0,  0, -4,  5,  5,  0,  0,  0, -6],
              [ 0,  0, -4,  0,  0, -5,  6,  6,  0]])
m, n = A.shape


## 1. Maximin: Alice (row player)


In [ ]:
p1 = cp.Variable(m, nonneg=True)
v1 = cp.Variable()  # Unrestricted in sign.
maximin = cp.Problem(cp.Maximize(v1),
                     [A.T @ p1 >= v1, cp.sum(p1) == 1])
maximin.solve(solver="SCIPY")
assert maximin.status == cp.OPTIMAL, maximin.status


## 2. Minimax: Bob (column player)


In [ ]:
p2 = cp.Variable(n, nonneg=True)
v2 = cp.Variable()  # Unrestricted in sign.
minimax = cp.Problem(cp.Minimize(v2),
                     [A @ p2 <= v2, cp.sum(p2) == 1])
minimax.solve(solver="SCIPY")
assert minimax.status == cp.OPTIMAL, minimax.status


## 3. Strategies, value, and a direct check


In [ ]:
print("Alice:", p1.value.round(4))
print("Bob:  ", p2.value.round(4))
print(f"Maximin: {v1.value:.6f}; minimax: {v2.value:.6f}")
gap = (A @ p2.value).max() - (A.T @ p1.value).min()
print(f"Best-response gap: {gap:.2e}")
assert abs(gap) < 1e-8
